# Milestone 2: Data Quality dan Cleaning

Lanjutan dari `01_data_profiling.ipynb`. Notebook ini memakai **DuckDB** untuk
profiling `SUMMARIZE`, kuantil, dan pemeriksaan duplikat, lalu memakai **Polars
(lazy)** untuk membersihkan data dan menyimpannya ke `data/processed/`.

**Dataset:** cuaca per jam 40 kota Indonesia, 2001-2024 (8,4 juta baris). Fokus
cleaning: missing values, outlier, duplikasi, dan validasi aturan fisik.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'cuaca_indonesia.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Jalankan `python src/download_data.py` lebih dahulu.'
    )

lazy_data = pl.scan_parquet(DATA_PATH)
row_count = lazy_data.select(pl.len()).collect().item()

print(f'Sumber : {DATA_PATH.name} ({DATA_PATH.stat().st_size / 1_000_000:.1f} MB)')
print(f'Baris  : {row_count:,}')

Sumber : cuaca_indonesia.parquet (190.2 MB)
Baris  : 8,415,360


## 1. Profiling dengan DuckDB `SUMMARIZE`

Ringkasan statistik seluruh kolom (tipe, null%, approx. distinct, min/max,
rata-rata, kuartil) dalam satu perintah SQL, dibaca langsung dari Parquet.

In [2]:
connection = duckdb.connect()

profil = pl.from_arrow(
    connection.sql(
        'SUMMARIZE SELECT * FROM read_parquet(?)', params=[str(DATA_PATH)]
    ).to_arrow_table()
)
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""waktu_lokal""","""TIMESTAMP""","""2001-01-01 00:00:00""","""2024-12-31 23:00:00""",204930,"""2012-12-31 23:30:00""",null,"""2007-01-11 17:05:22.183648""","""2012-12-22 05:23:05.2318""","""2018-12-28 22:08:36.163696""",8415360,0.00
"""T2M""","""DOUBLE""","""2.85""","""39.45""",3170,"""25.79794205833148""","""3.174262238450378""","""24.46986202689381""","""26.146263432224067""","""27.69681794051224""",8415360,0.00
"""T2MDEW""","""DOUBLE""","""-6.48""","""28.47""",2606,"""22.74126197215558""","""2.5280831215389568""","""22.068051852198664""","""23.41319090000426""","""24.225907982393533""",8415360,0.00
"""RH2M""","""DOUBLE""","""15.21""","""100.0""",7722,"""84.1328165485497""","""11.094820115886847""","""77.93454969412365""","""86.01679315851493""","""92.53059046269857""",8415360,0.00
"""QV2M""","""DOUBLE""","""3.12""","""24.95""",1812,"""17.807909977707453""","""1.8954380567345623""","""17.04679424266127""","""18.20024378047491""","""19.02359940203658""",8415360,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""provinsi""","""VARCHAR""","""Aceh""","""Sumatera Utara""",39,null,null,null,null,null,8415360,0.00
"""latitude""","""DOUBLE""","""-10.1772""","""5.5483""",41,"""-3.032562499996019""","""3.674864828956773""","""-6.180085151087135""","""-2.8139814880516423""","""-0.649986907643027""",8415360,0.00
"""longitude""","""DOUBLE""","""95.3238""","""140.669""",40,"""115.96157499998115""","""11.833684635349629""","""106.13598919238078""","""114.8652118529084""","""123.27203159445023""",8415360,0.00


## 2. Deteksi Missing Values

Hitung jumlah dan persentase null tiap kolom sebagai dasar keputusan penanganan.

In [3]:
null_summary = (
    lazy_data.select(pl.all().null_count())
    .collect()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns((pl.col('jumlah_null') / row_count * 100).round(4).alias('persen_null'))
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""waktu_lokal""",0,0.0
"""T2M""",0,0.0
"""T2MDEW""",0,0.0
"""RH2M""",0,0.0
"""QV2M""",0,0.0
…,…,…
"""provinsi""",0,0.0
"""latitude""",0,0.0
"""longitude""",0,0.0


## 3. Penanganan Missing Values

Seluruh kolom memiliki **0 nilai null**. Missing value sudah ditangani pada
tahap pengunduhan: *fill value* `-999` dari NASA POWER dinormalisasi menjadi
`null` di `src/download_data.py`. Karena tidak ada nilai hilang, **tidak
diperlukan imputasi**.

Kebijakan bila muncul null baru ke depan (mis. setelah agregasi): isi kolom
numerik dengan **median per kolom** dan kategorik dengan label `'tidak diketahui'`.
Langkah di bawah memverifikasi bahwa tidak ada null yang perlu diisi.

In [4]:
total_null = int(null_summary['jumlah_null'].sum())
kolom_bermasalah = null_summary.filter(pl.col('jumlah_null') > 0)

print(f'Total nilai null di seluruh kolom : {total_null:,}')
print(f'Kolom dengan null                 : {kolom_bermasalah.height}')
print('Kesimpulan: tidak ada imputasi yang diperlukan.')

# Kebijakan imputasi (tidak dijalankan karena tidak ada null) untuk reproduksibilitas:
# lazy_data_filled = lazy_data.with_columns(
#     pl.col(c).fill_null(pl.col(c).median()) for c in kolom_numerik
# )

Total nilai null di seluruh kolom : 0
Kolom dengan null                 : 0
Kesimpulan: tidak ada imputasi yang diperlukan.


## 4. Deteksi dan Penanganan Outlier

Deteksi outlier dengan pagar **IQR** (`Q1 - 1.5*IQR`, `Q3 + 1.5*IQR`) untuk
tiap kolom numerik. Kuantil dihitung di DuckDB.

In [5]:
kolom_numerik = [
    'T2M', 'T2MDEW', 'RH2M', 'QV2M', 'PRECTOTCORR',
    'PS', 'WS10M', 'WD10M', 'ALLSKY_SFC_SW_DWN', 'CLOUD_AMT',
]

fence_sql = ', '.join(
    f'quantile_cont("{c}", 0.25) AS "{c}_q1", quantile_cont("{c}", 0.75) AS "{c}_q3"'
    for c in kolom_numerik
)
fences = connection.execute(
    f'SELECT {fence_sql} FROM read_parquet(?)', [str(DATA_PATH)]
).fetchone()

rows = []
for index, kolom in enumerate(kolom_numerik):
    q1, q3 = fences[2 * index], fences[2 * index + 1]
    iqr = q3 - q1
    rows.append({
        'kolom': kolom,
        'q1': round(q1, 3),
        'q3': round(q3, 3),
        'iqr': round(iqr, 3),
        'batas_bawah': round(q1 - 1.5 * iqr, 3),
        'batas_atas': round(q3 + 1.5 * iqr, 3),
    })

count_sql = ', '.join(
    f'SUM(CASE WHEN "{r["kolom"]}" < {r["batas_bawah"]} '
    f'OR "{r["kolom"]}" > {r["batas_atas"]} THEN 1 ELSE 0 END) AS "{r["kolom"]}"'
    for r in rows
)
counts = connection.execute(
    f'SELECT {count_sql} FROM read_parquet(?)', [str(DATA_PATH)]
).fetchone()

outlier_table = pl.DataFrame(rows).with_columns(
    pl.Series('jumlah_outlier', list(counts), dtype=pl.Int64),
    (pl.Series('jumlah_outlier', list(counts), dtype=pl.Float64) / row_count * 100)
    .round(3)
    .alias('persen_outlier'),
)
outlier_table

kolom,q1,q3,iqr,batas_bawah,batas_atas,jumlah_outlier,persen_outlier
str,f64,f64,f64,f64,f64,i64,f64
"""T2M""",24.47,27.7,3.23,19.625,32.545,343329,4.08
"""T2MDEW""",22.07,24.23,2.16,18.83,27.47,484426,5.756
"""RH2M""",77.92,92.53,14.61,56.005,114.445,189737,2.255
"""QV2M""",17.05,19.02,1.97,14.095,21.975,449359,5.34
"""PRECTOTCORR""",0.01,0.28,0.27,-0.395,0.685,881543,10.475
"""PS""",98.43,100.55,2.12,95.25,103.73,1050525,12.483
"""WS10M""",1.21,3.3,2.09,-1.925,6.435,166757,1.982
"""WD10M""",107.3,255.7,148.4,-115.3,478.3,0,0.0
"""ALLSKY_SFC_SW_DWN""",0.0,418.42,418.42,-627.63,1046.05,171,0.002


**Keputusan penanganan.** Untuk data iklim, nilai di luar pagar IQR umumnya
adalah **ekstrem cuaca yang sah** (hujan sangat lebat, angin kencang), bukan
kesalahan pencatatan. Winsorisasi pada pagar IQR akan menghapus sinyal nyata
(mis. memotong `PRECTOTCORR` hingga ~0,5 mm/jam), sehingga **tidak dilakukan**.

Sebagai gantinya, outlier statistik dipertahankan dan diperbaiki hanya bila
**melanggar batas fisik**:

| Kolom | Aturan koreksi |
|---|---|
| `T2MDEW` | tidak boleh melebihi `T2M`: `T2MDEW = min(T2MDEW, T2M)` |
| `RH2M`, `CLOUD_AMT` | dijepit ke rentang `[0, 100]` |
| `PRECTOTCORR`, `WS10M` | dijepit ke `>= 0` |
| `WD10M` | dijepit ke `[0, 360]` |
| `PS` | dijepit ke rentang fisis `[60, 110]` kPa |

Transformasi ini diterapkan secara *lazy* di bawah dan hanya mengubah baris yang
benar-benar melanggar aturan.

In [6]:
lazy_clean = lazy_data.with_columns(
    pl.min_horizontal('T2MDEW', 'T2M').alias('T2MDEW'),
    pl.col('RH2M').clip(0, 100).alias('RH2M'),
    pl.col('CLOUD_AMT').clip(0, 100).alias('CLOUD_AMT'),
    pl.col('PRECTOTCORR').clip(lower_bound=0).alias('PRECTOTCORR'),
    pl.col('WS10M').clip(lower_bound=0).alias('WS10M'),
    pl.col('WD10M').clip(0, 360).alias('WD10M'),
    pl.col('PS').clip(60, 110).alias('PS'),
)

koreksi = lazy_data.select(
    pl.col('T2MDEW').gt(pl.col('T2M')).sum().alias('T2MDEW_dikoreksi'),
    (pl.col('RH2M') < 0).or_(pl.col('RH2M') > 100).sum().alias('RH2M_dikoreksi'),
    (pl.col('PRECTOTCORR') < 0).sum().alias('PRECTOTCORR_dikoreksi'),
    (pl.col('WS10M') < 0).sum().alias('WS10M_dikoreksi'),
).collect()
koreksi

T2MDEW_dikoreksi,RH2M_dikoreksi,PRECTOTCORR_dikoreksi,WS10M_dikoreksi
u32,u32,u32,u32
47859,0,0,0


## 5. Deduplikasi

Kunci unik alami dataset adalah `(kota, waktu_lokal)`: satu observasi per jam
per kota. Duplikat diperiksa di DuckDB sebelum memutuskan penghapusan.

In [7]:
jumlah_duplikat = connection.execute(
    '''
    SELECT COUNT(*) FROM (
        SELECT kota, waktu_lokal, COUNT(*) AS c
        FROM read_parquet(?)
        GROUP BY 1, 2
        HAVING c > 1
    )
    ''',
    [str(DATA_PATH)],
).fetchone()[0]

print(f'Duplikat pada kunci (kota, waktu_lokal): {jumlah_duplikat:,}')
if jumlah_duplikat == 0:
    print('Tidak ada duplikat; tidak ada baris yang dihapus.')
    lazy_dedup = lazy_clean
else:
    lazy_dedup = lazy_clean.unique(subset=['kota', 'waktu_lokal'], keep='first')
    print('Duplikat dihapus dengan mempertahankan kemunculan pertama.')

Duplikat pada kunci (kota, waktu_lokal): 0
Tidak ada duplikat; tidak ada baris yang dihapus.


## 6. Validasi Data Bersih

Jalankan aturan validasi pada data hasil cleaning. Semua pelanggaran harus
ber nilai **0**.

In [8]:
validasi = lazy_dedup.select(
    pl.len().alias('total_baris'),
    pl.col('T2MDEW').gt(pl.col('T2M')).sum().alias('pelanggaran_dewpoint'),
    (pl.col('RH2M') < 0).or_(pl.col('RH2M') > 100).sum().alias('pelanggaran_rh'),
    (pl.col('CLOUD_AMT') < 0).or_(pl.col('CLOUD_AMT') > 100).sum().alias('pelanggaran_cloud'),
    (pl.col('WD10M') < 0).or_(pl.col('WD10M') > 360).sum().alias('pelanggaran_arah_angin'),
    (pl.col('PRECTOTCORR') < 0).sum().alias('pelanggaran_presipitasi'),
    (pl.col('WS10M') < 0).sum().alias('pelanggaran_angin'),
).collect()
validasi

total_baris,pelanggaran_dewpoint,pelanggaran_rh,pelanggaran_cloud,pelanggaran_arah_angin,pelanggaran_presipitasi,pelanggaran_angin
u32,u32,u32,u32,u32,u32,u32
8415360,0,0,0,0,0,0


## 7. Simpan Data Bersih

Tulis hasil cleaning ke `data/processed/` dengan `sink_parquet` (streaming)
agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [9]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
lazy_dedup.sink_parquet(OUTPUT_PATH)

print(f'Ukuran awal : {row_count:,} baris')
print(f'Ukuran bersih: {pl.scan_parquet(OUTPUT_PATH).select(pl.len()).collect().item():,} baris')
print(f'Tersimpan di: {OUTPUT_PATH}')

Ukuran awal : 8,415,360 baris
Ukuran bersih: 8,415,360 baris
Tersimpan di: /home/fatih/Documents/Projects/tugas1-abd-2026/data/processed/dataset_bersih.parquet


## Catatan Milestone 2

**Ringkasan eksekusi.** Dataset dibaca 8.415.360 baris x 17 kolom (190,2 MB) dan hasil bersih disimpan ke `data/processed/dataset_bersih.parquet` dengan jumlah baris yang sama (tidak ada baris yang dibuang).

**1. Missing values.** Tidak ada nilai null pada seluruh kolom (0%). *Fill value* `-999` sudah dinormalisasi menjadi `null` pada tahap pengunduhan, sehingga **tidak ada imputasi** yang diperlukan.

**2. Outlier (pagar IQR 1.5x).** Persentase baris di luar pagar IQR per kolom:

| Kolom | Batas bawah | Batas atas | Jumlah outlier | Persen |
|---|---|---|---|---|
| T2M | 19,625 | 32,545 | 343.329 | 4,08% |
| T2MDEW | 18,830 | 27,470 | 484.426 | 5,76% |
| RH2M | 56,005 | 114,445 | 189.737 | 2,26% |
| QV2M | 14,095 | 21,975 | 449.359 | 5,34% |
| PRECTOTCORR | -0,395 | 0,685 | 881.543 | 10,48% |
| PS | 95,250 | 103,730 | 1.050.525 | 12,48% |
| WS10M | -1,925 | 6,435 | 166.757 | 1,98% |
| WD10M | -115,3 | 478,3 | 0 | 0,00% |
| ALLSKY_SFC_SW_DWN | -627,63 | 1.046,05 | 171 | 0,002% |
| CLOUD_AMT | -2,605 | 160,715 | 0 | 0,00% |

**3. Keputusan outlier.** Winsorisasi pada pagar IQR **ditolak** karena pagar tersebut memotong nilai iklim yang sah (mis. membatasi `PRECTOTCORR` di 0,685 mm/jam akan membuang 10,5% kejadian hujan, dan pagar `PS`/`RH2M` mengikuti variasi ketinggian antar kota). Sebagai gantinya, outlier statistik **dipertahankan** dan hanya nilai yang **melanggar batas fisik** yang dikoreksi:

- `T2MDEW = min(T2MDEW, T2M)` — memperbaiki 47.859 baris (0,57%) yang titik embunnya melebihi suhu (supersaturasi).
- `RH2M`, `CLOUD_AMT` dijepit ke `[0, 100]`; `PRECTOTCORR`, `WS10M` dijepit ke `>= 0`; `WD10M` ke `[0, 360]`; `PS` ke `[60, 110]` kPa — semuanya **0 perubahan** (data sudah dalam rentang fisis).

**4. Duplikasi.** Kunci alami `(kota, waktu_lokal)` bersifat unik: **0 duplikat**, sehingga tidak ada baris yang dihapus.

**5. Validasi akhir.** Seluruh aturan fisik lulus dengan **0 pelanggaran** pada data bersih (dewpoint, RH, cloud, arah angin, presipitasi, angin).

**Kesimpulan.** Dimensi kualitas paling bermasalah adalah **distribusi ekor (outlier)** pada `PS`, `PRECTOTCORR`, dan `QV2M`, bukan missing value atau duplikasi. Satu-satunya koreksi nilai yang berdampak adalah konsistensi titik embun terhadap suhu. Data bersih siap dipakai pada Milestone 3.